In [1]:
import json
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# load cleaned KB
all_chunks = []
with open("../data/processed/kb_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        all_chunks.append(json.loads(line))
print("Loaded chunks:", len(all_chunks))

# load splits
def load_pairs(name):
    pairs = []
    with open(f"../data/processed/qa_pairs_{name}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            pairs.append(json.loads(line))
    return pairs

train_pairs = load_pairs("train")
dev_pairs = load_pairs("dev")
test_pairs = load_pairs("test")
print(f"Pairs: train={len(train_pairs)}, dev={len(dev_pairs)}, test={len(test_pairs)}")

# load the fine-tuned retriever (Phase 2 output)
finetuned_model = SentenceTransformer("../data/indexes/retriever_finetuned_v1")
print("Loaded fine-tuned retriever. Device:", finetuned_model.device)

# re-encode the KB (fast — inference only, not training)
chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]
finetuned_embeddings = finetuned_model.encode(
    chunk_texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print("Chunk embeddings shape:", finetuned_embeddings.shape)

Loaded chunks: 4781
Pairs: train=19640, dev=2461, test=2468


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded fine-tuned retriever. Device: cuda:0


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Chunk embeddings shape: (4781, 384)


In [2]:
def get_top_k_candidates(model, chunk_embeddings, chunk_ids, pairs, k=20):
    queries = [p["query"] for p in pairs]
    query_embeddings = model.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    ranked_idx = np.argsort(-sims, axis=1)[:, :k]

    shortlists = []
    for i, pair in enumerate(pairs):
        candidate_ids = [chunk_ids[idx] for idx in ranked_idx[i]]
        shortlists.append({
            "query": pair["query"],
            "domain": pair["domain"],
            "gold_chunk_ids": pair["gold_chunk_ids"],
            "candidate_chunk_ids": candidate_ids,
        })
    return shortlists

train_shortlists = get_top_k_candidates(finetuned_model, finetuned_embeddings, chunk_ids, train_pairs, k=20)
dev_shortlists = get_top_k_candidates(finetuned_model, finetuned_embeddings, chunk_ids, dev_pairs, k=20)

print("Train shortlists:", len(train_shortlists))
print("Dev shortlists:", len(dev_shortlists))
print("\nExample:")
print("Query:", train_shortlists[0]["query"])
print("Gold:", train_shortlists[0]["gold_chunk_ids"])
print("Top-5 candidates:", train_shortlists[0]["candidate_chunk_ids"][:5])

Batches:   0%|          | 0/307 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Train shortlists: 19640
Dev shortlists: 2461

Example:
Query: Hello, I forgot o update my address, can you help me with that?
Gold: ['Top 5 DMV Mistakes and How to Avoid Them#3_0::chunk_2310']
Top-5 candidates: ['Top 5 DMV Mistakes and How to Avoid Them#3_0::chunk_2311', 'Escrow accounts#3_0::chunk_2844', 'View Your VA Payment History | Veterans Affairs#1_0::chunk_1833', 'How to replace a registration#1_0::chunk_2859', 'Information about transaction entries#3_0::chunk_3119']


In [3]:
def shortlist_position_stats(shortlists, k_values=(1, 5, 10, 20)):
    found_at = {k: 0 for k in k_values}
    not_found = 0
    for sl in shortlists:
        gold_set = set(sl["gold_chunk_ids"])
        candidates = sl["candidate_chunk_ids"]
        rank = next((r + 1 for r, cid in enumerate(candidates) if cid in gold_set), None)
        if rank is None:
            not_found += 1
        else:
            for k in k_values:
                if rank <= k:
                    found_at[k] += 1

    n = len(shortlists)
    print(f"Total dev queries: {n}")
    for k in k_values:
        print(f"  Gold chunk within top-{k}: {found_at[k]} ({found_at[k]/n:.2%})")
    print(f"  Gold chunk NOT in top-20 at all: {not_found} ({not_found/n:.2%})")

print("Retriever's own ranking (before any reranking):")
shortlist_position_stats(dev_shortlists)

Retriever's own ranking (before any reranking):
Total dev queries: 2461
  Gold chunk within top-1: 671 (27.27%)
  Gold chunk within top-5: 1346 (54.69%)
  Gold chunk within top-10: 1575 (64.00%)
  Gold chunk within top-20: 1767 (71.80%)
  Gold chunk NOT in top-20 at all: 694 (28.20%)


In [4]:
chunk_text_by_id = {c["chunk_id"]: c["text"] for c in all_chunks}

def build_reranker_training_pairs(shortlists, max_negatives_per_query=4):
    examples = []
    for sl in shortlists:
        gold_set = set(sl["gold_chunk_ids"])
        query = sl["query"]

        positives = [cid for cid in sl["candidate_chunk_ids"] if cid in gold_set]
        negatives = [cid for cid in sl["candidate_chunk_ids"] if cid not in gold_set]

        if not positives:
            continue  # gold wasn't retrieved at all — nothing to teach the reranker here

        for pos_id in positives[:1]:
            examples.append((query, chunk_text_by_id[pos_id], 1))

        for neg_id in negatives[:max_negatives_per_query]:
            examples.append((query, chunk_text_by_id[neg_id], 0))

    return examples

reranker_train_examples = build_reranker_training_pairs(train_shortlists, max_negatives_per_query=4)
print("Reranker training examples:", len(reranker_train_examples))

pos_count = sum(1 for _, _, label in reranker_train_examples if label == 1)
neg_count = sum(1 for _, _, label in reranker_train_examples if label == 0)
print(f"Positives: {pos_count}, Negatives: {neg_count}")

print("\nExample:")
print(reranker_train_examples[0])

Reranker training examples: 70070
Positives: 14014, Negatives: 56056

Example:
('Hello, I forgot o update my address, can you help me with that?', '1. Forgetting to Update Address\nBy statute , you must report a change of address to DMV within ten days of moving. That is the case for the address associated with your license, as well as all the addresses associated with each registered vehicle, which may differ.', 1)


In [5]:
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset as HFDataset

reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", num_labels=1)
print("Reranker model device:", reranker_model.model.device)

train_data = {
    "sentence1": [q for q, p, l in reranker_train_examples],
    "sentence2": [p for q, p, l in reranker_train_examples],
    "label": [float(l) for q, p, l in reranker_train_examples],
}
reranker_train_dataset = HFDataset.from_dict(train_data)
print("Reranker training examples loaded:", len(reranker_train_dataset))

loss = BinaryCrossEntropyLoss(reranker_model)

args = CrossEncoderTrainingArguments(
    output_dir="../data/indexes/reranker_finetuned_v1",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    logging_steps=200,
    save_strategy="no",
)

trainer = CrossEncoderTrainer(
    model=reranker_model,
    args=args,
    train_dataset=reranker_train_dataset,
    loss=loss,
)
trainer.train()

reranker_model.save("../data/indexes/reranker_finetuned_v1")
print("Reranker training complete, saved to data/indexes/reranker_finetuned_v1")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\Shrey\Desktop\TUF_F15_Data\Shrey_academics\Deep_Learning\Project_DL\Project_Attempt2\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shrey\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Reranker model device: cuda:0
Reranker training examples loaded: 70070


Step,Training Loss
200,0.757840
400,0.478441
600,0.458040
800,0.450827
1000,0.440874
1200,0.432278
1400,0.428326
1600,0.433278
1800,0.435695
2000,0.430796


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Reranker training complete, saved to data/indexes/reranker_finetuned_v1


In [6]:
def rerank_shortlists(reranker, shortlists, chunk_text_by_id):
    reranked = []
    for sl in shortlists:
        query = sl["query"]
        candidates = sl["candidate_chunk_ids"]
        pairs = [[query, chunk_text_by_id[cid]] for cid in candidates]
        scores = reranker.predict(pairs, show_progress_bar=False)
        order = np.argsort(-scores)
        new_candidates = [candidates[i] for i in order]
        reranked.append({**sl, "candidate_chunk_ids": new_candidates})
    return reranked

dev_reranked = rerank_shortlists(reranker_model, dev_shortlists, chunk_text_by_id)

print("AFTER reranking:")
shortlist_position_stats(dev_reranked)

print("\nBEFORE reranking (retriever's own order, for comparison):")
shortlist_position_stats(dev_shortlists)

AFTER reranking:
Total dev queries: 2461
  Gold chunk within top-1: 809 (32.87%)
  Gold chunk within top-5: 1386 (56.32%)
  Gold chunk within top-10: 1580 (64.20%)
  Gold chunk within top-20: 1767 (71.80%)
  Gold chunk NOT in top-20 at all: 694 (28.20%)

BEFORE reranking (retriever's own order, for comparison):
Total dev queries: 2461
  Gold chunk within top-1: 671 (27.27%)
  Gold chunk within top-5: 1346 (54.69%)
  Gold chunk within top-10: 1575 (64.00%)
  Gold chunk within top-20: 1767 (71.80%)
  Gold chunk NOT in top-20 at all: 694 (28.20%)
